# HW2 on H100 (Colab)

End-to-end: every PDF deliverable that needs a code run is covered. Each run writes to its own `logs/<category>/<ts>_<name>/` directory.

**Before you start:** Runtime → Change runtime type → **H100 GPU** → Save.

## 1. Setup

Clones, installs deps, then **kills the kernel on purpose** (vLLM downgrades torch — Python must re-import). Colab will say "Your session crashed" — that's expected. After it does, run cell 2.

In [ ]:
!git clone https://github.com/trevorbchen/148bhw2.git || (cd 148bhw2 && git pull)
%cd /content/148bhw2

# vLLM pins torch==2.5.1; install it first so it dictates the torch version, then
# install basics with --no-deps so it doesn't try to pull torch 2.6.
# transformers MUST be pinned: vLLM 0.7.2 reads tokenizer.all_special_tokens_extended
# which was removed in newer transformers releases.
!pip install -q vllm==0.7.2 transformers==4.48.3 datasets pandas pyarrow \
    latex2sympy2-extended "math-verify[antlr4-13-2]" pylatexenc==2.10 sympy \
    humanfriendly wandb einops einx jaxtyping tiktoken regex psutil submitit
!pip install -q -e ./basics --no-deps

import os
os.kill(os.getpid(), 9)  # force restart so the new torch is picked up

## 2. Systems benchmarks (Section 2.3-2.6)

In [ ]:
# Re-set CWD after the kernel restart from cell 1 (%cd is per-kernel state).
%cd /content/148bhw2

# §2.3 (2): forward / forward-backward / train-step across sizes.
for size in ["small", "medium", "large"]:
    for mode in ["forward", "forward-backward", "train-step"]:
        !python -m systems.benchmark --model-size {size} --mode {mode}

In [ ]:
# §2.3 (3): warmup sweep on the medium model — write-up compares 0/1/2/5/10 warmup steps.
for w in [0, 1, 2, 5, 10]:
    !python -m systems.benchmark --model-size medium --mode forward-backward \
        --warmup-steps {w} --measure-steps 10

In [ ]:
# §2.4: Nsight Systems profiling.
# Writes .nsys-rep files into logs/systems/nsys/. Open in the Nsight Systems
# desktop app (https://developer.nvidia.com/nsight-systems) to answer the
# 5 sub-questions (top kernel, softmax vs matmul, kernel mix in train step, etc.).
import os, shutil, glob

def _find_nsys():
    p = shutil.which('nsys')
    if p:
        return p
    cands = sorted(
        glob.glob('/usr/local/cuda*/bin/nsys')
        + glob.glob('/opt/nvidia/**/nsys', recursive=True)
        + glob.glob('/usr/local/bin/nsys')
    )
    return cands[-1] if cands else None

nsys = _find_nsys()
if nsys is None:
    print('nsys not found — installing newest nsight-systems-* via apt...')
    !apt-get update -q
    !apt-get install -y $(apt-cache search '^nsight-systems-' | sort -V | tail -1 | awk '{print $1}') \
        || apt-get install -y cuda-nsight-systems-12-4
    nsys = _find_nsys()
assert nsys, 'Could not locate nsys'
os.environ['PATH'] = os.path.dirname(nsys) + ':' + os.environ['PATH']
print('Using nsys:', nsys)
!nsys --version

!mkdir -p logs/systems/nsys
for size in ['small', 'medium', 'large']:
    for ctx in [32, 64, 128, 256]:
        for mode in ['forward', 'forward-backward', 'train-step']:
            tag = f'{size}_ctx{ctx}_{mode}'
            print(f'\n=== nsys: {tag} ===')
            !nsys profile -t cuda,nvtx,osrt --force-overwrite=true \
                -o logs/systems/nsys/{tag} \
                python -m systems.benchmark \
                --model-size {size} --context-length {ctx} --mode {mode} \
                --warmup-steps 1 --measure-steps 3 --nvtx 2>&1 | tail -10

In [ ]:
# §2.5 (3): mixed precision (bf16) — pairs with the §2.3 fp32 runs above.
for size in ["small", "medium", "large"]:
    !python -m systems.benchmark --model-size {size} --mode forward-backward --use-bf16

In [ ]:
# §2.6: memory profiler on the LARGE model (PDF p8) across context lengths.
# (1) two timelines: forward-only and full train-step (open the .pickle in https://pytorch.org/memory_viz)
# (2) peak memory per context length, fwd vs train-step (use peak_mem_mb in results.json)
# (3) bf16 mixed-precision memory comparison
for ctx in [128, 256, 512]:
    !python -m systems.benchmark --model-size large --mode forward \
        --context-length {ctx} --use-memory-profiler --measure-steps 3
    !python -m systems.benchmark --model-size large --mode train-step \
        --context-length {ctx} --use-memory-profiler --measure-steps 3
    !python -m systems.benchmark --model-size large --mode train-step \
        --context-length {ctx} --use-memory-profiler --use-bf16 --measure-steps 3

## 3. Attention benchmark (Section 2.7-2.8)

In [ ]:
# §2.7 + §2.8 (1): attention sweep, eager and torch.compile.
# Grid: head_dim ∈ {16,32,64,128} × seq_len ∈ {64,128,256,512,1024}, batch=8.
# OOM rows are tagged "error": "OOM" in results.json.
!python -m systems.attention_benchmark
!python -m systems.attention_benchmark --compile-attention

In [ ]:
# §2.8 (2): compile the WHOLE transformer (not just attention) and compare to vanilla.
for size in ["small", "medium", "large"]:
    for mode in ["forward", "forward-backward", "train-step"]:
        !python -m systems.benchmark --model-size {size} --mode {mode} --compile-model

## 4. Alignment baselines (Section 3.1-3.2)

First run downloads Qwen2.5-Math-1.5B (~3GB).  Defaults per PDF p13: `--split train`, reward-fn auto-picked per mode (`answer_tag` for direct, `r1_zero` for cot/self_consistency).

In [ ]:
!python -m alignment.eval --mode direct
!python -m alignment.eval --mode cot
!python -m alignment.eval --mode self_consistency --k 5

## 5. GRPO training (Section 3.5)

Two 50-step runs to compare normalize-by-std True vs False (PDF §3.5 problem grpo group standard deviation). Each is roughly an hour on H100.

In [ ]:
# §3.5 (grpo train loop): 50 iterations with normalize_by_std=True (default).
!python -m alignment.grpo \
    --n-grpo-steps 50 \
    --rollout-batch-size 32 \
    --group-size 8 \
    --train-batch-size 32 \
    --gradient-accumulation-steps 16 \
    --eval-every 5 \
    --n-eval-examples 256 \
    --run-name grpo_std_true

In [ ]:
# §3.5 (grpo group standard deviation): same config but normalize_by_std=False.
!python -m alignment.grpo \
    --n-grpo-steps 50 \
    --rollout-batch-size 32 \
    --group-size 8 \
    --train-batch-size 32 \
    --gradient-accumulation-steps 16 \
    --eval-every 5 \
    --n-eval-examples 256 \
    --no-normalize-by-std \
    --run-name grpo_std_false

In [ ]:
# §3.5 plot: validation rewards over steps for both runs.
# Saves logs/alignment/grpo_val_curves.png — include in writeup.
import json, glob
from pathlib import Path
import matplotlib.pyplot as plt

def _latest_run(name):
    matches = sorted(glob.glob(f'logs/alignment/*_{name}'))
    return Path(matches[-1]) if matches else None

def _curve(run_dir):
    steps, ans, fmt = [], [], []
    for line in (run_dir / 'train_log.jsonl').read_text().splitlines():
        row = json.loads(line)
        if 'val/answer_reward' in row:
            steps.append(row['step'])
            ans.append(row['val/answer_reward'])
            fmt.append(row['val/format_reward'])
    return steps, ans, fmt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for label, run_name in [('normalize_by_std=True', 'grpo_std_true'),
                         ('normalize_by_std=False', 'grpo_std_false')]:
    run = _latest_run(run_name)
    if run is None:
        print(f'skip {label}: no run found')
        continue
    s, a, f = _curve(run)
    axes[0].plot(s, a, marker='o', label=label)
    axes[1].plot(s, f, marker='o', label=label)

axes[0].set(xlabel='step', ylabel='val/answer_reward', title='GRPO: validation answer reward')
axes[1].set(xlabel='step', ylabel='val/format_reward', title='GRPO: validation format reward')
for ax in axes:
    ax.grid(alpha=0.3); ax.legend()

out = Path('logs/alignment/grpo_val_curves.png')
plt.tight_layout(); plt.savefig(out, dpi=120)
plt.show()
print(f'Saved {out}')

## 6. Bundle logs and download

Colab VMs are ephemeral — pull the logs back to your laptop before the session dies. Excludes saved model weights (`final/`) which are large and not needed for the writeup.

In [ ]:
!ls -R logs/ | head -60
!cd logs && zip -r /content/logs.zip . -x '*final/*'
from google.colab import files
files.download('/content/logs.zip')